### Abstract 

##### _Abstract overview of the notebook_

Since 2008, guests and hosts have used Airbnb to expand on traveling possibilities and present a more unique, personalized way of experiencing the world. Today, Airbnb became one of a kind service that is used and recognized by the whole world. Data analysis on millions of listings provided through Airbnb is a crucial factor for the company. These millions of listings generate a lot of data - data that can be analyzed and used for security, business decisions, understanding of customers' and providers' (hosts) behavior and performance on the platform, guiding marketing initiatives, implementation of innovative additional services and much more.

### Data Source

##### _Abstract overview of the notebook_

This dataset has around 49,000 observations in it with 16 columns and it is a mix between categorical and numeric values.

### Acquaring and Loading Data

##### _Presenting the code and methods for acquiring the data. Loading the data into appropriate format for analysis. Explaining the process and results_

In [1]:
# %load_ext cudf.pandas
import pandas as pd

In [3]:
airbnb = pd.read_csv("/home/dias-benchmarks/new_notebooks/nyc-airbnb/AB_NYC_2019.csv")
airbnb.head(3)

,id,name,host_id,host_name,neighbourhood_group,neighbourhood,latitude,longitude,room_type,price,minimum_nights,number_of_reviews,last_review,reviews_per_month,calculated_host_listings_count,availability_365
0,2539,Clean & quiet apt home by the park,2787,John,Brooklyn,Kensington,40.64749,-73.97237,Private room,149,1,9,2018-10-19,0.21,6,365
1,2595,Skylit Midtown Castle,2845,Jennifer,Manhattan,Midtown,40.75362,-73.98377,Entire home/apt,225,1,45,2019-05-21,0.38,2,355
2,3647,THE VILLAGE OF HARLEM....NEW YORK !,4632,Elisabeth,Manhattan,Harlem,40.80902,-73.94190,Private room,150,3,0,NaN,NaN,1,365


In [4]:
len(airbnb)

48895

In [5]:
factor = 100
airbnb = pd.concat([airbnb] * factor)

In [6]:
airbnb.dtypes

id                                  int64
name                               object
host_id                             int64
host_name                          object
neighbourhood_group                object
neighbourhood                      object
latitude                          float64
longitude                         float64
room_type                          object
price                               int64
minimum_nights                      int64
number_of_reviews                   int64
last_review                        object
reviews_per_month                 float64
calculated_host_listings_count      int64
availability_365                    int64
dtype: object

After loading the dataset in and from the head of AB_2019_NYC dataset we can see a number of things. These 16 columns provide a very rich amount of information for deep data exploration we can do on this dataset. We do already see some missing values, which will require cleaning and handling of NaN values. Later, we may need to continue with mapping certain values to ones and zeros for predictive analytics.

### Understadning, Wrangling and Cleaning Data

##### _Presenting the code and methods for acquiring the data. Loading the data into appropriate format for analysis. Explaining the process and results_

In [7]:
airbnb.isnull().sum()

id                                      0
name                                 1600
host_id                                 0
host_name                            2100
neighbourhood_group                     0
neighbourhood                           0
latitude                                0
longitude                               0
room_type                               0
price                                   0
minimum_nights                          0
number_of_reviews                       0
last_review                       1005200
reviews_per_month                 1005200
calculated_host_listings_count          0
availability_365                        0
dtype: int64

In our case, missing data that is observed does not need too much special treatment. Looking into the nature of our dataset we can state further things: columns "name" and "host_name" are irrelevant and insignificant to our data analysis, columns "last_review" and "review_per_month" need very simple handling. To elaborate, "last_review" is date; if there were no reviews for the listing - date simply will not exist. In our case, this column is irrelevant and insignificant therefore appending those values is not needed. For "review_per_month" column we can simply append it with 0.0 for missing values; we can see that in "number_of_review" that column will have a 0, therefore following this logic with 0 total reviews there will be 0.0 rate of reviews per month. Therefore, let's proceed with removing columns that are not important and handling of missing data.

In [8]:
airbnb.drop(["id", "host_name", "last_review"], axis=1, inplace=True)
airbnb.head(3)

,name,host_id,neighbourhood_group,neighbourhood,latitude,longitude,room_type,price,minimum_nights,number_of_reviews,reviews_per_month,calculated_host_listings_count,availability_365
0,Clean & quiet apt home by the park,2787,Brooklyn,Kensington,40.64749,-73.97237,Private room,149,1,9,0.21,6,365
1,Skylit Midtown Castle,2845,Manhattan,Midtown,40.75362,-73.98377,Entire home/apt,225,1,45,0.38,2,355
2,THE VILLAGE OF HARLEM....NEW YORK !,4632,Manhattan,Harlem,40.80902,-73.94190,Private room,150,3,0,NaN,1,365


Please note that we are dropping'host_name' not only because it is insignificant but also for ethical reasons. There should be no reasoning to continue data exploration and model training (which we will be doing later) towards specific individuals based on their names. Why is that? Those names are assigned to actual humans, also they present no security threat or military/governmental interest based on the nature of the dataset, therefore names are unimportant to us.

In [9]:
airbnb.fillna({"reviews_per_month": 0}, inplace=True)
airbnb.reviews_per_month.isnull().sum()

0

In [10]:
airbnb.neighbourhood_group.unique()

array(['Brooklyn', 'Manhattan', 'Queens', 'Staten Island', 'Bronx'],
      dtype=object)

In [11]:
len(airbnb.neighbourhood.unique())

221

In [12]:
airbnb.room_type.unique()

array(['Private room', 'Entire home/apt', 'Shared room'], dtype=object)

Understanding unique values and categorical data that we have in our dataset was the last step we had to do. It looks like for those columns' values we will be doing some mapping to prepare the dataset for predictive analysis.

### Exploring and Visualizing Data

##### Exploring the data by analyzing its statistics and visualizing the values of features and correlations between different features. Explaining the process and the results

Now that we are ready for an exploration of our data, we can make a rule that we are going to be working from left to right. The reason some may prefer to do this is due to its set approach - some datasets have a big number of attributes, plus this way we will remember to explore each column individually to make sure we learn as much as we can about our dataset.

In [13]:
top_host = airbnb.host_id.value_counts().head(10)
top_host

host_id
219517861    32700
107434423    23200
30283594     12100
137358866    10300
16098958      9600
12243051      9600
61391963      9100
22541573      8700
200380610     6500
7503643       5200
Name: count, dtype: int64

In [14]:
top_host_check = airbnb.calculated_host_listings_count.max()
top_host_check

327

In [15]:
# sns.set(rc={'figure.figsize': (10, 8)})
# sns.set_style('white')

In [16]:
top_host_df = pd.DataFrame(top_host)
top_host_df.reset_index(inplace=True)
top_host_df.rename(columns={"index": "Host_ID", "host_id": "P_Count"}, inplace=True)
top_host_df

,P_Count,count
0,219517861,32700
1,107434423,23200
2,30283594,12100
3,137358866,10300
4,16098958,9600
5,12243051,9600
6,61391963,9100
7,22541573,8700
8,200380610,6500
9,7503643,5200


Interesting, we can see that there is a good distribution between top 10 hosts with the most listings. First host has more than 300+ listings.

In [17]:
sub_1 = airbnb.loc[airbnb["neighbourhood_group"] == "Brooklyn"]
price_sub1 = sub_1[["price"]]
sub_2 = airbnb.loc[airbnb["neighbourhood_group"] == "Manhattan"]
price_sub2 = sub_2[["price"]]
sub_3 = airbnb.loc[airbnb["neighbourhood_group"] == "Queens"]
price_sub3 = sub_3[["price"]]
sub_4 = airbnb.loc[airbnb["neighbourhood_group"] == "Staten Island"]
price_sub4 = sub_4[["price"]]
sub_5 = airbnb.loc[airbnb["neighbourhood_group"] == "Bronx"]
price_sub5 = sub_5[["price"]]
price_list_by_n = [price_sub1, price_sub2, price_sub3, price_sub4, price_sub5]

In [18]:
p_l_b_n_2 = []
nei_list = ["Brooklyn", "Manhattan", "Queens", "Staten Island", "Bronx"]
for x in price_list_by_n:
    i = x.describe(percentiles=[0.25, 0.5, 0.75])
    i = i.iloc[3:]
    i.reset_index(inplace=True)
    i.rename(columns={"index": "Stats"}, inplace=True)
    p_l_b_n_2.append(i)
p_l_b_n_2[0].rename(columns={"price": nei_list[0]}, inplace=True)
p_l_b_n_2[1].rename(columns={"price": nei_list[1]}, inplace=True)
p_l_b_n_2[2].rename(columns={"price": nei_list[2]}, inplace=True)
p_l_b_n_2[3].rename(columns={"price": nei_list[3]}, inplace=True)
p_l_b_n_2[4].rename(columns={"price": nei_list[4]}, inplace=True)
stat_df = p_l_b_n_2
stat_df = [df.set_index("Stats") for df in stat_df]
stat_df = stat_df[0].join(stat_df[1:])
stat_df

,Brooklyn,Manhattan,Queens,Staten Island,Bronx
Stats,,,,,
min,0.0,0.0,10.0,13.0,0.0
25%,60.0,95.0,50.0,50.0,45.0
50%,90.0,150.0,75.0,75.0,65.0
75%,150.0,220.0,110.0,110.0,99.0
max,10000.0,10000.0,10000.0,5000.0,2500.0


In [19]:
sub_6 = airbnb[airbnb.price < 500]
sub_6
# viz_2 = sns.violinplot(data=sub_6, x='neighbourhood_group', y='price')
# viz_2.set_title('Density and distribution of prices for each neighberhood_group')

,name,host_id,neighbourhood_group,neighbourhood,latitude,longitude,room_type,price,minimum_nights,number_of_reviews,reviews_per_month,calculated_host_listings_count,availability_365
0,Clean & quiet apt home by the park,2787,Brooklyn,Kensington,40.64749,-73.97237,Private room,149,1,9,0.21,6,365
1,Skylit Midtown Castle,2845,Manhattan,Midtown,40.75362,-73.98377,Entire home/apt,225,1,45,0.38,2,355
2,THE VILLAGE OF HARLEM....NEW YORK !,4632,Manhattan,Harlem,40.80902,-73.94190,Private room,150,3,0,0.00,1,365
3,Cozy Entire Floor of Brownstone,4869,Brooklyn,Clinton Hill,40.68514,-73.95976,Entire home/apt,89,1,270,4.64,1,194
4,Entire Apt: Spacious Studio/Loft by central park,7192,Manhattan,East Harlem,40.79851,-73.94399,Entire home/apt,80,10,9,0.10,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
48890,Charming one bedroom - newly renovated rowhouse,8232441,Brooklyn,Bedford-Stuyvesant,40.67853,-73.94995,Private room,70,2,0,0.00,2,9
48891,Affordable room in Bushwick/East Williamsburg,6570630,Brooklyn,Bushwick,40.70184,-73.93317,Private room,40,4,0,0.00,2,36
48892,Sunny Studio at Historical Neighborhood,23492952,Manhattan,Harlem,40.81475,-73.94867,Entire home/apt,115,10,0,0.00,1,27
48893,43rd St. Time Square-cozy single bed,30985759,Manhattan,Hell's Kitchen,40.75751,-73.99112,Shared room,55,1,0,0.00,6,2


Great, with a statistical table and a violin plot we can definitely observe a couple of things about distribution of prices for Airbnb in NYC boroughs. First, we can state that Manhattan has the highest range of prices for the listings with \\$150 price as average observation, followed by Brooklyn with \\$90 per night. Queens and Staten Island appear to have very similar distributions, Bronx is the cheapest of them all. This distribution and density of prices were completely expected; for example, as it is no secret that Manhattan is one of the most expensive places in the world to live in, where Bronx on other hand appears to have lower standards of living.

In [20]:
airbnb.neighbourhood.value_counts().head(10)

neighbourhood
Williamsburg          392000
Bedford-Stuyvesant    371400
Harlem                265800
Bushwick              246500
Upper West Side       197100
Hell's Kitchen        195800
East Village          185300
Upper East Side       179800
Crown Heights         156400
Midtown               154500
Name: count, dtype: int64

In [21]:
sub_7 = airbnb.loc[
    airbnb["neighbourhood"].isin(
        [
            "Williamsburg",
            "Bedford-Stuyvesant",
            "Harlem",
            "Bushwick",
            "Upper West Side",
            "Hell's Kitchen",
            "East Village",
            "Upper East Side",
            "Crown Heights",
            "Midtown",
        ]
    )
]
sub_7
# viz_3 = sns.catplot(x='neighbourhood', hue='neighbourhood_group', col='room_type', data=sub_7, kind='count')
# viz_3.set_xticklabels(rotation=90)

,name,host_id,neighbourhood_group,neighbourhood,latitude,longitude,room_type,price,minimum_nights,number_of_reviews,reviews_per_month,calculated_host_listings_count,availability_365
1,Skylit Midtown Castle,2845,Manhattan,Midtown,40.75362,-73.98377,Entire home/apt,225,1,45,0.38,2,355
2,THE VILLAGE OF HARLEM....NEW YORK !,4632,Manhattan,Harlem,40.80902,-73.94190,Private room,150,3,0,0.00,1,365
6,BlissArtsSpace!,7356,Brooklyn,Bedford-Stuyvesant,40.68688,-73.95596,Private room,60,45,49,0.40,1,0
7,Large Furnished Room Near B'way,8967,Manhattan,Hell's Kitchen,40.76489,-73.98493,Private room,79,2,430,3.47,1,220
8,Cozy Clean Guest Room - Family Apt,7490,Manhattan,Upper West Side,40.80178,-73.96723,Private room,79,2,118,0.99,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
48890,Charming one bedroom - newly renovated rowhouse,8232441,Brooklyn,Bedford-Stuyvesant,40.67853,-73.94995,Private room,70,2,0,0.00,2,9
48891,Affordable room in Bushwick/East Williamsburg,6570630,Brooklyn,Bushwick,40.70184,-73.93317,Private room,40,4,0,0.00,2,36
48892,Sunny Studio at Historical Neighborhood,23492952,Manhattan,Harlem,40.81475,-73.94867,Entire home/apt,115,10,0,0.00,1,27
48893,43rd St. Time Square-cozy single bed,30985759,Manhattan,Hell's Kitchen,40.75751,-73.99112,Shared room,55,1,0,0.00,6,2


Amazing, but let' breakdown on what we can see from this plot. First, we can see that our plot consists of 3 subplots - that is the power of using catplot; with such output, we can easily proceed with comparing distributions among interesting attributes. Y and X axes stay exactly the same for each subplot, Y-axis represents a count of observations and X-axis observations we want to count. However, there are 2 more important elements: column and hue; those 2 differentiate subplots. After we specify the column and determined hue we are able to observe and compare our Y and X axes among specified column as well as color-coded. So, what do we learn from this? The observation that is definitely contrasted the most is that 'Shared room' type Airbnb listing is barely available among 10 most listing-populated neighborhoods. Then, we can see that for these 10 neighborhoods only 2 boroughs are represented: Manhattan and Brooklyn; that was somewhat expected as Manhattan and Brooklyn are one of the most traveled destinations, therefore would have the most listing availability. We can also observe that Bedford-Stuyvesant and Williamsburg are the most popular for Manhattan borough, and Harlem for Brooklyn.

In [22]:
# viz_4 = sub_6.plot(kind='scatter', x='longitude', y='latitude', label='availability_365', c='price', cmap=plt.get_cmap('jet'), colorbar=True, alpha=0.4, figsize=(10, 8))
# viz_4.legend()

Good, scatterplot worked just fine to output our latitude and longitude points. However, it would be nice to have a map bellow for fully immersive heatmap in ourcase - let's see what we can do!

Fantastic! After scaling our image the best we can, we observe that we end up with a very immersive heatmap. Using latitude and longitude points were able to visualize all NYC listings. Also, we added a color-coded range for each point on the map based on the price of the listing. However, it is important to note that we had to drop some extremely high values as they are treated as outliers for our analysis. 

In [23]:
names_ = []
for name in airbnb.name:
    names_.append(name)


def split_name(name):
    spl = str(name).split()
    return spl


names_for_count_ = []
for x in names_:
    for word in split_name(x):
        word = word.lower()
        names_for_count_.append(word)

In [24]:
from collections import Counter

top_25_w = Counter(names_for_count_).most_common()
top_25_w = top_25_w[0:25]

In [25]:
sub_w = pd.DataFrame(top_25_w)
sub_w.rename(columns={0: "Words", 1: "Count"}, inplace=True)

In [26]:
# viz_5 = sns.barplot(x='Words', y='Count', data=sub_w)
# viz_5.set_title('Counts of the top 25 used words for listing names')
# viz_5.set_ylabel('Count of words')
# viz_5.set_xlabel('Words')
# viz_5.set_xticklabels(viz_5.get_xticklabels(), rotation=80)

We can observe that finding out and going over top 25 used listings' name words - we are able to see one clear trend. It shows that hosts are simply describing their listing in a short form with very specific terms for easier search by a potential traveler. Such wors are 'room', 'bedroom', 'private', 'apartment', 'studio'. This shows that there are no catchphrases or 'popular/trending' terms that are used for names; hosts use very simple terms describing the space and the area where the listing is. This technique was somewhat expected as dealing with multilingual customers can be tricky and you definitely want to describe your space in a concise and understood form as much as possible.

In [27]:
top_reviewed_listings = airbnb.nlargest(10, "number_of_reviews")
top_reviewed_listings

,name,host_id,neighbourhood_group,neighbourhood,latitude,longitude,room_type,price,minimum_nights,number_of_reviews,reviews_per_month,calculated_host_listings_count,availability_365
11759,Room near JFK Queen Bed,47621202,Queens,Jamaica,40.6673,-73.76831,Private room,47,1,629,14.58,2,333
11759,Room near JFK Queen Bed,47621202,Queens,Jamaica,40.6673,-73.76831,Private room,47,1,629,14.58,2,333
11759,Room near JFK Queen Bed,47621202,Queens,Jamaica,40.6673,-73.76831,Private room,47,1,629,14.58,2,333
11759,Room near JFK Queen Bed,47621202,Queens,Jamaica,40.6673,-73.76831,Private room,47,1,629,14.58,2,333
11759,Room near JFK Queen Bed,47621202,Queens,Jamaica,40.6673,-73.76831,Private room,47,1,629,14.58,2,333
11759,Room near JFK Queen Bed,47621202,Queens,Jamaica,40.6673,-73.76831,Private room,47,1,629,14.58,2,333
11759,Room near JFK Queen Bed,47621202,Queens,Jamaica,40.6673,-73.76831,Private room,47,1,629,14.58,2,333
11759,Room near JFK Queen Bed,47621202,Queens,Jamaica,40.6673,-73.76831,Private room,47,1,629,14.58,2,333
11759,Room near JFK Queen Bed,47621202,Queens,Jamaica,40.6673,-73.76831,Private room,47,1,629,14.58,2,333
11759,Room near JFK Queen Bed,47621202,Queens,Jamaica,40.6673,-73.76831,Private room,47,1,629,14.58,2,333


In [28]:
top_reviewed_listings.price.mean()

47.0

There is no reason to visualize this as table format would be the most suitable output for better reading. From this table output, we can observe that top 10 most reviewed listings on Airbnb for NYC has price average of \\$65 with most of the listings under \\$50, and 9/10 of them are 'Private room' type; top reviewed listing has 629 reviews.

### Conclusion

##### _Summarizing our findings, suggesting other features_